# 01 - Exploratory Data Analysis

This notebook performs exploratory data analysis on the MIDI dataset.

In [ ]:
import sys
from pathlib import Path
import importlib.util

# Add the src directory to path
src_dir = Path.cwd().parent / 'src'
sys.path.insert(0, str(src_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Dynamically import modules
def import_module_from_file(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module

# Import config and helpers
config = import_module_from_file('config', src_dir / 'utils' / 'config.py')
helpers = import_module_from_file('helpers', src_dir / 'utils' / 'helpers.py')
midi_loader = import_module_from_file('midi_loader', src_dir / 'preprocessing' / 'midi_loader.py')

# Get the functions we need
COMPOSERS = config.COMPOSERS
RAW_DATA_DIR = config.RAW_DATA_DIR
METADATA_FILE = config.METADATA_FILE
FIGURES_DIR = config.FIGURES_DIR
FIGURE_DPI = config.FIGURE_DPI

set_random_seed = helpers.set_random_seed
plot_class_distribution = helpers.plot_class_distribution
load_midi_dataset = midi_loader.load_midi_dataset

set_random_seed()
print("EDA notebook ready!")

In [ ]:
# Load dataset
metadata, stats = load_midi_dataset(RAW_DATA_DIR, COMPOSERS, save_metadata=True)
metadata_df = pd.read_csv(METADATA_FILE)
print(f"Loaded {len(metadata_df)} MIDI files")

In [ ]:
# Basic statistics
print("Dataset Statistics:")
print(metadata_df.describe())

In [ ]:
# Class distribution
plot_class_distribution(metadata_df['label'].values, COMPOSERS, "Class Distribution")

In [ ]:
# Feature distributions
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
metadata_df['duration'].hist(bins=50, ax=axes[0,0])
axes[0,0].set_title('Duration Distribution')
metadata_df['num_notes'].hist(bins=50, ax=axes[0,1])
axes[0,1].set_title('Note Count Distribution')
metadata_df['tempo'].hist(bins=50, ax=axes[1,0])
axes[1,0].set_title('Tempo Distribution')
for composer in COMPOSERS:
    metadata_df[metadata_df['composer']==composer]['duration'].hist(bins=30, alpha=0.5, label=composer, ax=axes[1,1])
axes[1,1].legend()
axes[1,1].set_title('Duration by Composer')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_distributions.png', dpi=FIGURE_DPI)
plt.show()